# Оцінка даних lingbow + Модель A vs B

**Що це:** оцінка якості оброблених вибірок (`data/processed`) і вимір приросту
Моделі B (з сигналами першої доби) над Моделлю A (лише ознаки до публікації).

_Згенеровано асистентом, червень 2026._

## 0. Контекст і обмеження середовища

- Аналіз зроблено на вже **оброблених** вибірках lingbow (`train/valid/test.parquet`).
- Сирий lingbow тут недоступний: у пісочниці **Hugging Face заблоковано** (proxy 403),
  а сирі таблиці на диску не збереглися. Тому повний EDA «з нуля» і математичне
  порівняння міток (breakout, приріст підписників) робиться окремо, після локальної
  докачки (`scripts/download_raw_lingbow.py`).
- Мітка тут: **within-creator ER > медіана автора (train)** на горизонті 14 днів.

## 1. Завантаження і баланс класів

In [1]:
import os, json, warnings
import pandas as pd, numpy as np
warnings.filterwarnings('ignore')
BASE = 'data/processed' if os.path.exists('data/processed') else '../data/processed'
tr = pd.read_parquet(f'{BASE}/train.parquet')
va = pd.read_parquet(f'{BASE}/valid.parquet')
te = pd.read_parquet(f'{BASE}/test.parquet')
meta = json.load(open(f'{BASE}/feature_meta.json'))
A = meta['A_COLS']; B = meta['A_COLS'] + meta['B_EXTRA']
print('shapes train/valid/test:', tr.shape, va.shape, te.shape)
print('y mean train/valid/test:', round(tr.y.mean(),4), round(va.y.mean(),4), round(te.y.mean(),4))
print('label:', meta['label'])
print('horizon H (days):', meta['horizon_H'], '| day-1 cutoff:', meta['d1'])

shapes train/valid/test: (144579, 26) (30848, 26) (31152, 26)
y mean train/valid/test: 0.5025 0.4868 0.4715
label: within-creator ER > author train-median @ day H
horizon H (days): 14 | day-1 cutoff: 1


## 2. Якість вибірок: пропуски, викиди, тип спліту

Перевіряємо: чи є пропуски, як обрізані хвости (winsorize) і чи спліт часовий
(автори повторюються) чи creator-disjoint.

In [2]:
miss = tr[B].isna().mean().sort_values(ascending=False)
print('макс. частка пропусків серед ознак:', round(miss.max(),4))
ov = len(set(tr.author_id) & set(te.author_id))
print(f'автори: train={tr.author_id.nunique()} test={te.author_id.nunique()} перетин={ov}')
print('=> спліт ЧАСОВИЙ (автори повторюються в часі), не creator-disjoint;')
print('   отже для узагальнення на контент треба ще міряти leave-one-creator-out.')
tr[['duration_s','char_len','n_hashtags','er_d1','log_play_d1']].describe(percentiles=[.5,.95,.99]).round(2)

макс. частка пропусків серед ознак: 0.0
автори: train=1863 test=1726 перетин=1717
=> спліт ЧАСОВИЙ (автори повторюються в часі), не creator-disjoint;
   отже для узагальнення на контент треба ще міряти leave-one-creator-out.


,duration_s,char_len,n_hashtags,er_d1,log_play_d1
count,144579.00,144579.00,144579.00,144579.00,144579.00
mean,36.88,115.82,5.91,0.10,6.58
std,35.18,118.38,6.58,0.07,1.61
min,5.09,0.00,0.00,0.00,3.40
50%,20.77,85.00,4.00,0.09,6.02
95%,116.37,347.00,18.00,0.24,10.14
99%,165.20,664.00,39.00,0.33,12.00
max,165.20,664.00,39.00,0.34,12.00


## 3. Модель A (до публікації) vs Модель B (+ перша доба)

Однакова мітка, однаковий тест. A — лише ознаки до публікації; B — ті самі + сигнали дня 1.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
med = tr[B].median()
def fit_eval(cols, model):
    Xtr, Xte = tr[cols].fillna(med), te[cols].fillna(med)
    if model=='logreg':
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    else:
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.06, max_iter=200, l2_regularization=1.0)
    m.fit(Xtr, tr.y); p = m.predict_proba(Xte)[:,1]
    return roc_auc_score(te.y,p), average_precision_score(te.y,p)
rows=[]
for name,cols in [('A — до публікації',A),('B — + доба',B)]:
    for mdl in ['logreg','hgb']:
        au,ap=fit_eval(cols,mdl); rows.append([name,mdl,round(au,3),round(ap,3)])
au,ap=fit_eval(meta['B_EXTRA'],'hgb'); rows.append(['лише сигнали дня 1','hgb',round(au,3),round(ap,3)])
print('TEST base rate:', round(te.y.mean(),3))
pd.DataFrame(rows, columns=['модель','алгоритм','ROC-AUC','PR-AUC'])

TEST base rate: 0.471


,модель,алгоритм,ROC-AUC,PR-AUC
0,A — до публікації,logreg,0.537,0.499
1,A — до публікації,hgb,0.554,0.514
2,B — + доба,logreg,0.825,0.790
3,B — + доба,hgb,0.838,0.807
4,лише сигнали дня 1,hgb,0.835,0.804


## 4. Висновки і застереження

- **Модель A ≈ випадкова** (AUC ~0.55): з підпису, тривалості й часу успіх майже
  не передбачається. Це природа задачі (стеля pre-publication), а не баг.
- **Модель B сильна** (AUC ~0.84), але майже весь сигнал — це рання залученість
  (`er_d1`), що корелює з фінальною міткою майже за побудовою. Тобто це радше
  «рання ER передбачає фінальну ER», ніж глибока майстерність — але операційно
  рішення на добу реальне і корисне.
- **Бізнес-висновок:** чистий передпублікаційний оракул тут не працює; цінність —
  у рішенні «після першої доби: підсилювати чи зрізати».

## 5. Що заблоковано / наступні кроки

- Математичне порівняння міток (breakout = перегляди/підписники; within-creator ER;
  приріст підписників) потребує **сирих** `engagement_daily` і `creator_daily`.
- Докачати локально: `python scripts/download_raw_lingbow.py` → `data/raw/lingbow/`.
- Далі: EDA з нуля + чистка, порівняння міток (base rate, fame-leak, кореляції,
  learnability контентом), якісні спліти без витоку, і вже тоді — тренування A/B.